# Curated Transactions and Internal Joins

This notebook communicates the data-quality outcome of the automated curation stage. Run the shared pipeline first:

```bash
python -m src.run_pipeline --data-root tables --output-root data/curated
```

The source tables are not modified. Invalid transactions, if found, are written to a quarantine file; merchant-master non-matches remain in the curated transaction table and are explicitly flagged.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd

candidate_roots = [
    Path(os.environ.get('PROJECT2_CURATED_ROOT', '')),
    Path.cwd() / 'data' / 'curated',
    Path.cwd().parent / 'data' / 'curated',
]
OUTPUT_ROOT = next((path for path in candidate_roots if (path / 'curation_audit.csv').is_file()), None)
if OUTPUT_ROOT is None:
    raise FileNotFoundError('Run src.run_pipeline first, or set PROJECT2_CURATED_ROOT.')

audit = pd.read_csv(OUTPUT_ROOT / 'curation_audit.csv')
join_coverage = pd.read_csv(OUTPUT_ROOT / 'join_coverage.csv')
snapshot_coverage = pd.read_csv(OUTPUT_ROOT / 'snapshot_coverage.csv')
merchant_exceptions = pd.read_csv(OUTPUT_ROOT / 'merchant_match_exceptions.csv')
print(f'Loaded curation artefacts from: {OUTPUT_ROOT}')

## 1. What happened to the transactions?

The audit makes each cleaning decision visible. Transactions are quarantined only for invalid keys, dates or amounts, or for duplicate-order conflicts; high-value transactions are flagged rather than deleted.

In [ ]:
display(audit)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(audit['stage'], audit['output_rows'], color='#2F6B9A')
ax.set(title='Transactions retained through curation', ylabel='Transaction rows')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
plt.show()

## 2. Are the internal joins safe?

User and consumer joins are required to preserve the transaction row count. The merchant join is deliberately a left join: missing merchant-master records are retained for follow-up rather than removed from merchant-performance analysis.

In [ ]:
display(join_coverage)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(join_coverage['join_name'], join_coverage['match_rate'] * 100, color='#3B8C88')
ax.set(xlim=(0, 100), xlabel='Match rate (%)', title='Internal join coverage')
for index, value in enumerate(join_coverage['match_rate'] * 100):
    ax.text(value + 0.8, index, f'{value:.2f}%', va='center')
fig.tight_layout()
plt.show()

print(f"Unmatched merchant ABNs: {len(merchant_exceptions):,}")
print(f"Transactions retained with an unmatched merchant: {merchant_exceptions['transaction_count'].sum():,}")

## 3. What is the real timeline?

Snapshot folder names are not treated as authoritative. This table derives each snapshot's actual coverage from its daily transaction partitions, which is the date basis passed to downstream fraud, ABS and ranking work.

In [ ]:
display(snapshot_coverage)

top_exceptions = merchant_exceptions.nlargest(10, 'transaction_count')
display(top_exceptions)